# H1 — Job Title, Country & Seniority vs. Salary

## Hypothesis

        Job title explains less about salary than where the company is and how senior the role is, and analyses that rank titles without these controls are misleading.**

### Process
1. Load the six annual surveys (2020-2025) and extract the columns needed for H1.
2. Standardise column names across years (they differ slightly).
3. Clean & engineer features (seniority bucket, country tier, log-salary).
4. Save a single merged `H1_clean.csv` ready for modelling.


In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 80)

## 1 · File paths and column maps

Each year uses slightly different column names for the same concept.

In [3]:
BASE = "/Users/aditya/Documents/PR/PR FInal Project/2026Spring_projects/Data"          
YEAR_FILES = {
    yr: "{base}/stack-overflow-developer-survey-{year}/survey_results_public.csv".format(
        base=BASE, year=yr
    )
    for yr in range(2020, 2026)
}

FINAL_COLS = [
    "ResponseId", "MainBranch", "Employment", "Country",
    "DevType", "OrgSize", "YearsCode", "YearsCodePro",
    "WorkExp", "ICorPM", "ConvertedCompYearly",
]

COLUMN_OVERRIDES = {
    2020: {
        "ResponseId":"Respondent",    
        "ConvertedCompYearly": "ConvertedComp", 
        "WorkExp":None,
        "ICorPM":None,             
    },
    2021: {
        "WorkExp": None,   
        "ICorPM":  None,   
    },
    2025: {
        "YearsCodePro": None,  
    },
}

print("Paths and column maps defined.")
print("Years covered : {}".format(list(YEAR_FILES.keys())))


Paths and column maps defined.
Years covered : [2020, 2021, 2022, 2023, 2024, 2025]


## 2 · Load each year, extract & rename columns

In [4]:
def build_col_map(year):
    """
    Build a column mapping for year.
    Start with the identity mapping for all FINAL_COLS, then apply any overrides
    for this year.
    """
    col_map = {col: col for col in FINAL_COLS}
    col_map.update(COLUMN_OVERRIDES.get(year, {}))
    return col_map


def load_year(year):
    """Load one survey CSV, extract H1 columns, rename to Final Column names."""
    path = YEAR_FILES[year]
    col_map = build_col_map(year)

    if not os.path.exists(path):
        print("{}: file not found -> {}".format(year, path))
        return pd.DataFrame()

    # Read only the columns that actually exist in this file
    raw_cols_needed = [v for v in col_map.values() if v is not None]
    df_raw = pd.read_csv(path, usecols=lambda c: c in raw_cols_needed)

    # Build the canonical dataframe
    df = pd.DataFrame()
    for canonical, raw in col_map.items():
        if raw is not None and raw in df_raw.columns:
            df[canonical] = df_raw[raw]
        else:
            df[canonical] = np.nan   # column missing for this year -> NaN column

    df.insert(0, "Year", year)
    present = [c for c, r in col_map.items() if r is not None]
    print("{}: {:>6,} rows loaded | cols present: {}".format(year, len(df), present))
    return df


print("Loading all years...")
frames = [load_year(yr) for yr in sorted(YEAR_FILES)]

raw_combined = pd.concat([f for f in frames if not f.empty], ignore_index=True)
print("Combined shape : {}".format(raw_combined.shape))
raw_combined.head(3)


Loading all years...
2020: 64,461 rows loaded | cols present: ['ResponseId', 'MainBranch', 'Employment', 'Country', 'DevType', 'OrgSize', 'YearsCode', 'YearsCodePro', 'ConvertedCompYearly']
2021: 83,439 rows loaded | cols present: ['ResponseId', 'MainBranch', 'Employment', 'Country', 'DevType', 'OrgSize', 'YearsCode', 'YearsCodePro', 'ConvertedCompYearly']
2022: 73,268 rows loaded | cols present: ['ResponseId', 'MainBranch', 'Employment', 'Country', 'DevType', 'OrgSize', 'YearsCode', 'YearsCodePro', 'WorkExp', 'ICorPM', 'ConvertedCompYearly']
2023: 89,184 rows loaded | cols present: ['ResponseId', 'MainBranch', 'Employment', 'Country', 'DevType', 'OrgSize', 'YearsCode', 'YearsCodePro', 'WorkExp', 'ICorPM', 'ConvertedCompYearly']
2024: 65,437 rows loaded | cols present: ['ResponseId', 'MainBranch', 'Employment', 'Country', 'DevType', 'OrgSize', 'YearsCode', 'YearsCodePro', 'WorkExp', 'ICorPM', 'ConvertedCompYearly']
2025: 49,191 rows loaded | cols present: ['ResponseId', 'MainBranch', '

,Year,ResponseId,MainBranch,Employment,Country,DevType,OrgSize,YearsCode,YearsCodePro,WorkExp,ICorPM,ConvertedCompYearly
0,2020,1,I am a developer by profession,"Independent contractor, freelancer, or self-employed",Germany,"Developer, desktop or enterprise applications;Developer, full-stack",2 to 9 employees,36,27,NaN,NaN,NaN
1,2020,2,I am a developer by profession,Employed full-time,United Kingdom,"Developer, full-stack;Developer, mobile","1,000 to 4,999 employees",7,4,NaN,NaN,NaN
2,2020,3,I code primarily as a hobby,NaN,Russian Federation,NaN,NaN,4,NaN,NaN,NaN,NaN


In [5]:
audit_cols = ["ConvertedCompYearly","DevType","Country",
              "YearsCode","YearsCodePro","WorkExp","ICorPM","OrgSize"]

print("=== Row counts per year ===")
print(raw_combined.groupby("Year").size().to_string())


=== Row counts per year ===
Year
2020    64461
2021    83439
2022    73268
2023    89184
2024    65437
2025    49191


## 4 · Filter — keep professional developers only

MainBranch values that indicate a professional developer vary slightly acrossyears but all contain *"developer by profession"*.  

Drop students and the not-employed to avoid mixing compensation types.


In [6]:
print("Unique MainBranch values:")
print(raw_combined["MainBranch"].value_counts().to_string())

Unique MainBranch values:
MainBranch
I am a developer by profession                                                                313764
I code primarily as a hobby                                                                    22326
I am not primarily a developer, but I write code sometimes as part of my work/studies          20359
I am a student who is learning to code                                                         19999
I am not primarily a developer, but I write code sometimes as part of my work                  17874
I am learning to code                                                                          17730
I used to be a developer by profession, but no longer am                                        8412
None of these                                                                                   3221
I work with developers or my work supports developers but am not a developer by profession       996


We can extract 2 values for further analysis:
* I am a developer by profession 
* developer by profession

In [7]:
# Keep rows where MainBranch indicates professional developer
is_prof = raw_combined["MainBranch"].str.contains("developer by profession")
print("number of professional developers:", is_prof.sum())

number of professional developers: 323172


In [8]:
print("\nUnique Employment values:")
print(raw_combined["Employment"].value_counts().head(20).to_string())


Unique Employment values:
Employment
Employed, full-time                                                         135751
Employed full-time                                                           98854
Independent contractor, freelancer, or self-employed                         37321
Employed                                                                     33750
Student, full-time                                                           30676
Student                                                                      12215
Not employed, but looking for work                                           12029
Employed, full-time;Independent contractor, freelancer, or self-employed     11397
Employed part-time                                                            4678
Student, part-time                                                            4341
Employed, part-time                                                           4042
Not employed, and not looking for work           

Keep only the ones who are EMployed Full time or Part time or freelancer

In [9]:
KEEP_EMPLOYMENT = [
    "Employed, full-time",
    "Employed full-time",
    "Independent contractor, freelancer, or self-employed",
    "Self-employed",
    "Employed part-time",
    "Employed, part-time",
]
is_employed = raw_combined["Employment"].isin(KEEP_EMPLOYMENT)
print("number of rows with employment: {}".format(is_employed.sum()))

number of rows with employment: 280646


In [10]:
df_prof = raw_combined[is_prof & is_employed].copy()
print(f"Rows after professional-dev filter : {len(df_prof):,}  "
      f"(dropped {len(raw_combined)-len(df_prof):,})")
print(df_prof.groupby("Year").size().to_string())

Rows after professional-dev filter : 247,248  (dropped 177,732)
Year
2020    45951
2021    55912
2022    43525
2023    55775
2024    40393
2025     5692


## 5 · Salary cleaning - ASSUMPTION

ConvertedCompYearly is already in USD across all years.  
We apply bounds to remove data-entry errors:

* Lower bound : $5,000 below this is too low for annual compensation  
* Upper bound : $500,000  above this are outliers that skew regressions  

In [ ]:
df_prof["ConvertedCompYearly"] = pd.to_numeric(df_prof["ConvertedCompYearly"])

print("Salary coverage before bounds filter:")
print(df_prof.groupby("Year")["ConvertedCompYearly"].agg(count="count", min="min", max="max", mean="mean").round(0)
      .join(
        df_prof.groupby("Year")["ConvertedCompYearly"]
        .apply(lambda x: x.isnull().sum())
        .to_frame("null_count")
    )
      .to_string())

SALARY_LOW  = 5_000
SALARY_HIGH = 500_000

salary_mask = (df_prof["ConvertedCompYearly"].between(SALARY_LOW, SALARY_HIGH))
df_salary = df_prof[salary_mask].copy()

print(f"\nRows after salary filter : {len(df_salary):,}  "
      f"(dropped {len(df_prof)-len(df_salary):,} out-of-range salaries)")

print(df_salary.groupby("Year")["ConvertedCompYearly"].agg(
    count="count", min="min", max="max", mean="mean"
).round(0).to_string())

Salary coverage before bounds filter:
      count  min         max      mean  null_count
Year                                              
2020  31550  0.0   2000000.0  102704.0       14401
2021  42703  1.0  30468516.0  116479.0       13209
2022  32072  1.0  50000000.0  170138.0       11453
2023  40713  1.0  74351432.0  102258.0       15062
2024  18156  1.0   9000000.0   89198.0       22237
2025   2701  1.0   6890299.0  103363.0        2991

Rows after salary filter : 157,207  (dropped 90,041 out-of-range salaries)
      count     min       max     mean
Year                                  
2020  28949  5000.0  500000.0  69641.0
2021  40162  5000.0  500000.0  72738.0
2022  29751  5000.0  500000.0  86385.0
2023  38890  5000.0  500000.0  95138.0
2024  17002  5000.0  500000.0  88816.0
2025   2453  5000.0  500000.0  96135.0


## 6 · Seniority feature engineering

### 6a. YearsCode & YearsCodePro
Both columns store free-text values like Less than 1 year and
More than 50 years. We convert them to numbers.

We also notice a few values which are numeric are also string


In [12]:
mask = df_salary["YearsCode"].apply(lambda x: not pd.isna(x) and not isinstance(x, (int, float, np.integer, np.floating)))
print(df_salary.loc[mask, "YearsCode"].unique())

['17' '8' '10' '7' '5' '37' '9' '30' '4' '19' '20' '25' '16' '36' '6' '43'
 '23' '11' '38' '24' '21' '3' '40' '15' '27' '12' '46' '13' '14' '33' '22'
 '18' '28' '32' '44' '26' '42' '34' '2' '31' '35' '1' '41' '29'
 'More than 50 years' '47' '39' 'Less than 1 year' '45' '50' '48' '49']


In [13]:
def clean_years(series: pd.Series) -> pd.Series:
    """Convert YearsCode / YearsCodePro text to numeric."""
    s = series.astype(str).str.strip()
    s = s.replace({
        "Less than 1 year": "0.5",
        "More than 50 years": "51",
        "nan": np.nan,
        "NA": np.nan, 
    })
    return pd.to_numeric(s)


df_salary["YearsCode"]    = clean_years(df_salary["YearsCode"])
df_salary["YearsCodePro"] = clean_years(df_salary["YearsCodePro"])
df_salary["WorkExp"]      = pd.to_numeric(df_salary["WorkExp"])

print("YearsCode stats:")
print(df_salary.groupby("Year")["YearsCode"].agg(count="count", median="median", mean="mean").round(1).to_string())

print("\nYearsCodePro stats:")
print(df_salary.groupby("Year")["YearsCodePro"].agg(count="count", median="median", mean="mean").round(1).to_string())

YearsCode stats:
      count  median  mean
Year                     
2020  28848    12.0  14.5
2021  39865    12.0  14.5
2022  29700    12.0  14.4
2023  38841    13.0  15.8
2024  16972    13.0  15.9
2025   2447    20.0  21.8

YearsCodePro stats:
      count  median  mean
Year                     
2020  28870     6.0   8.7
2021  39815     7.0   9.1
2022  29658     7.0   9.6
2023  38761     8.0  10.9
2024  16961     8.0  10.9
2025      0     NaN   NaN


In [14]:
df_salary.shape

(157207, 13)

### 6b. Unified seniority proxy - ASSUMPTION

We can use YearsCodePro but year 2025 doesnt have any values. Hence we can use WorkExp as an alternate field for 2025. 

To make analysis easier, we can put into different categories


In [15]:
# referred to this doc to place into bins: https://pandas.pydata.org/docs/reference/api/pandas.cut.html
df_salary["SeniorityYears"] = df_salary["YearsCodePro"].fillna(df_salary["YearsCode"])

bins = [0, 2, 5, 10, 20, float("inf")]
labels = ["0-2 yrs", "3-5 yrs", "6-10 yrs", "11-20 yrs", "20+ yrs"]

# Assign buckets in one line
df_salary["SeniorityBucket"] = pd.cut(df_salary["SeniorityYears"], bins=bins, labels=labels)

In [16]:
df_salary["SeniorityBucket"].value_counts().sort_index()

SeniorityBucket
0-2 yrs      22699
3-5 yrs      37276
6-10 yrs     42779
11-20 yrs    35992
20+ yrs      18126
Name: count, dtype: int64

In [17]:
df_salary['DevType'].head(10)

7     Developer, back-end;Developer, desktop or enterprise applications;Developer,...
9                      Database administrator;Developer, full-stack;Developer, mobile
10    Developer, back-end;Developer, desktop or enterprise applications;Developer,...
11                                                      Designer;Developer, front-end
13                 Developer, desktop or enterprise applications;Developer, front-end
15    Database administrator;Developer, back-end;Developer, desktop or enterprise ...
16                                                              Developer, full-stack
18                                                              Developer, full-stack
23                                                                Developer, back-end
25                                                              Developer, full-stack
Name: DevType, dtype: str

In [18]:
# Primary title = first value before the first semicolon
df_salary["DevTypePrimary"] = (
    df_salary["DevType"]
    .astype(str)
    .str.split(";")
    .str[0]
    .str.strip()
    .replace("nan", np.nan)
)
print("Top 20 primary DevType values (all years combined):")
print(df_salary["DevTypePrimary"].value_counts().head(20).to_string())

Top 20 primary DevType values (all years combined):
DevTypePrimary
Developer, full-stack                            46503
Developer, back-end                              36708
Developer, front-end                             22448
Developer, mobile                                 9841
Developer, desktop or enterprise applications     9035
Data scientist or machine learning specialist     4404
Developer, embedded applications or devices       3339
Engineer, data                                    3099
Database administrator                            2342
Other (please specify):                           2069
Academic researcher                               1976
Engineering manager                               1965
Data or business analyst                          1804
DevOps specialist                                 1607
Designer                                          1502
Developer, game or graphics                        993
Engineer, site reliability                         95

In [19]:
df_salary['DevTypePrimary'].unique()

array(['Developer, back-end', 'Database administrator', 'Designer',
       'Developer, desktop or enterprise applications',
       'Developer, full-stack', 'Data or business analyst',
       'Developer, front-end',
       'Data scientist or machine learning specialist',
       'Developer, mobile', nan, 'DevOps specialist',
       'Academic researcher', 'Developer, QA or test',
       'Developer, embedded applications or devices',
       'Senior executive/VP', 'Developer, game or graphics', 'Educator',
       'Engineer, data', 'Engineering manager',
       'Engineer, site reliability', 'System administrator', 'Scientist',
       'Product manager', 'Other (please specify):',
       'Senior Executive (C-Suite, VP, etc.)', 'Student',
       'Marketing or sales professional', 'Cloud infrastructure engineer',
       'Security professional', 'Blockchain', 'Project manager',
       'Research & Development role', 'Developer Experience',
       'Developer Advocate', 'Hardware Engineer', 'Develop

In [20]:
# Simplified category map
DEV_CATEGORY_MAP = {
    "Developer, full-stack":"Full-Stack",
    "Developer, back-end":"Back-End",
    "Developer, front-end":"Front-End",
    "Developer, mobile":"Mobile",
    "Data scientist or machine learning specialist": "Data/ML",
    "Data or business analyst":"Data/ML",
    "Engineer, data":"Data/ML",
    "DevOps specialist":"DevOps/Cloud",
    "Engineer, site reliability":"DevOps/Cloud",
    "Cloud infrastructure engineer":"DevOps/Cloud",
    "Developer, embedded applications or devices": "Embedded/Systems",
    "Engineer, embedded systems":"Embedded/Systems",
    "Security professional":"Security",
    "Developer, QA or test":"QA/Test",
    "Product manager":"Management",
    "Engineering manager":"Management",
    "Senior Executive (C-Suite, VP, etc.)":"Management",
    "Developer, desktop or enterprise applications": "Desktop/Enterprise",
    "Academic researcher":"Research/Academia",
    "Scientist":"Research/Academia",
}

df_salary["DevCategory"] = (
    df_salary["DevTypePrimary"]
    .map(DEV_CATEGORY_MAP)
    .fillna("Other")
)

print("\nDevCategory distribution:")
print(df_salary["DevCategory"].value_counts().to_string())



DevCategory distribution:
DevCategory
Full-Stack            46503
Back-End              36708
Front-End             22448
Other                 10414
Mobile                 9841
Data/ML                9307
Desktop/Enterprise     9035
DevOps/Cloud           3383
Embedded/Systems       3339
Management             3034
Research/Academia      2171
QA/Test                 861
Security                163


## 8 · Country tier

For regression we need a manageable location variable.  
We group countries into **4 salary tiers** based on typical developer compensation:


* Tier 1 — High 

    USA, Switzerland, Australia, Canada, Israel, UK, Germany, Netherlands, Sweden, Denmark, Norway, Finland, Austria, Ireland, New Zealand, Singapore

* Tier 2 — Upper-Mid

    France, Japan, South Korea, Spain, Italy, UAE, Poland, Czech Republic, Portugal, Brazil (tech hubs), others in W. Europe

* Tier 3 — Mid 

    India, Brazil, Mexico, Turkey, Argentina, South Africa, Romania, Hungary, Ukraine

* Tier 4 — Lower

    All remaining countries


In [21]:
print(f"\nUnique countries in data : {df_salary['Country'].nunique()}")
print("Top 20 countries by count:")
print(df_salary["Country"].value_counts().head(20).to_string())
print(f"\nUnique countries: {list(df_salary['Country'].unique())}")


Unique countries in data : 185
Top 20 countries by count:
Country
United States of America                                29481
Germany                                                 12059
India                                                    8986
United Kingdom of Great Britain and Northern Ireland     8973
United States                                            6834
Canada                                                   6485
France                                                   6009
Brazil                                                   4837
Netherlands                                              4079
Poland                                                   3957
Spain                                                    3849
Australia                                                3524
Italy                                                    3420
Sweden                                                   3161
Russian Federation                                       2195
Uni

Using GPT, we seggregated countries into different tiers based on developers compnesation in dollars.

Link for Chat: https://chatgpt.com/share/69ec2267-2378-83ea-a08c-b7797e2d7a4a

In [22]:
TIER1 = {
        'United States',
        'United States of America',
        'Switzerland',
        'Australia',
        'Canada',
        'Israel',
        'Denmark',
        'Norway',
        'Netherlands',
        'Singapore',
        'Luxembourg'
        }
TIER2 = {
        'United Kingdom',
        'United Kingdom of Great Britain and Northern Ireland',
        'Germany',
        'Ireland',
        'Sweden',
        'Finland',
        'Austria',
        'France',
        'New Zealand',
        'United Arab Emirates',
        'Japan',
        'Iceland',
        'Belgium',
        }
TIER3 = {
        'Spain',
        'Italy',
        'Portugal',
        'Czech Republic',
        'Poland',
        'Hungary',
        'Romania',
        'Estonia',
        'Slovenia',
        'Spain',
        'Italy',
        'Portugal',
        'Czech Republic',
        'Poland',
        'Hungary',
        'Romania',
        'Estonia',
        'Slovenia',
        'Slovakia',
        'Lithuania',
        'Latvia',
        'Greece',
        'Turkey',
        'Cyprus',
        'Malta',
        'Croatia',
        'Serbia',
        'Bulgaria',
        'Ukraine',
        'Russian Federation',
        'Kazakhstan',
        'China',
        'South Korea',
        'Republic of Korea',
        'Taiwan',
        'Malaysia',
        'Thailand',
        'Chile',
        'Mexico',
        'Brazil',
        'Argentina',
        'Costa Rica',
        'Panama',
        'Uruguay',
        'South Africa',
        'Saudi Arabia',
        'Qatar',
        'Kuwait'
}

def assign_tier(country: str) -> str:
    if pd.isna(country):
        return "Unknown"
    if country in TIER1:
        return "Tier1_High"
    if country in TIER2:
        return "Tier2_UpperMid"
    if country in TIER3:
        return "Tier3_Mid"
    return "Tier4_Lower"

df_salary["CountryTier"] = df_salary["Country"].apply(assign_tier)

print("Country tier distribution:")
print(df_salary["CountryTier"].value_counts().to_string())

Country tier distribution:
CountryTier
Tier1_High        57709
Tier2_UpperMid    39988
Tier3_Mid         39896
Tier4_Lower       19614


In [23]:
df_salary.shape

(157207, 18)

## 9 · OrgSize cleaning

Standardise the free-text org-size buckets into an ordered numeric scale.


In [24]:
print("Unique OrgSize values:")
print(df_salary["OrgSize"].value_counts().to_string())

Unique OrgSize values:
OrgSize
20 to 99 employees                                    33871
100 to 499 employees                                  30334
10,000 or more employees                              20488
1,000 to 4,999 employees                              17773
2 to 9 employees                                      13739
10 to 19 employees                                    13256
500 to 999 employees                                  10915
Just me - I am a freelancer, sole proprietor, etc.     6475
5,000 to 9,999 employees                               6346
I don’t know                                           1670
Less than 20 employees                                  255


In [25]:
# check for empty or null OrgSize values
mask = df_salary["OrgSize"].isnull() | (df_salary["OrgSize"].
str.strip() == "")
print(f"Number of rows with empty or null OrgSize: {mask.sum()}")

Number of rows with empty or null OrgSize: 2085


In [26]:
ORG_SIZE_MAP = {"Just me - I am a freelancer, sole proprietor, etc.": 1,
    "2 to 9 employees":                                   2,
    "10 to 19 employees":                                 3,
    "20 to 99 employees":                                 4,
    "100 to 499 employees":                               5,
    "500 to 999 employees":                               6,
    "1,000 to 4,999 employees":                           7,
    "5,000 to 9,999 employees":                           8,
    "10,000 or more employees":                           9,
    
    "Less than 20 employees":                             3,  # grouped with 10-19 range
    "I don't know":                                       None}

df_salary["OrgSizeNum"] = df_salary["OrgSize"].map(ORG_SIZE_MAP)

print("OrgSizeNum distribution:")
print(df_salary["OrgSizeNum"].value_counts().sort_index().to_string())
# number of empty or null OrgSize values after mapping
num_null_orgsize = df_salary["OrgSizeNum"].isnull().sum()
print(f"Number of rows with empty or null OrgSizeNum: {num_null_orgsize}")


OrgSizeNum distribution:
OrgSizeNum
1.0     6475
2.0    13739
3.0    13511
4.0    33871
5.0    30334
6.0    10915
7.0    17773
8.0     6346
9.0    20488
Number of rows with empty or null OrgSizeNum: 3755


Originally Empty: 2085

Cell with values 'I don't know': 1670

Hence 3755 are empty

---
## 11 · Final H1 dataset

Keep only the canonical columns needed for H1 analysis.  
Drop rows missing the two most critical fields: `ConvertedCompYearly` and `DevType`.


In [27]:
df_salary.columns

Index(['Year', 'ResponseId', 'MainBranch', 'Employment', 'Country', 'DevType',
       'OrgSize', 'YearsCode', 'YearsCodePro', 'WorkExp', 'ICorPM',
       'ConvertedCompYearly', 'LogSalary', 'SeniorityYears', 'SeniorityBucket',
       'DevTypePrimary', 'DevCategory', 'CountryTier', 'OrgSizeNum'],
      dtype='str')

In [28]:
print("Final columns in H1 dataset:")
print(df_salary.columns.to_list())

df_h1 = df_salary.copy()

# Drop rows missing salary OR devtype (both are essential for H1)
before = len(df_h1)
df_h1 = df_h1.dropna(subset=["ConvertedCompYearly", "DevTypePrimary"])
after = len(df_h1)
print(f"Dropped {before - after:,} rows missing salary or DevType.")
print(f"\nFinal H1 dataset shape: {df_h1.shape}")
print(f"\nRows per year:")
print(df_h1.groupby("Year").size().to_string())

Final columns in H1 dataset:
['Year', 'ResponseId', 'MainBranch', 'Employment', 'Country', 'DevType', 'OrgSize', 'YearsCode', 'YearsCodePro', 'WorkExp', 'ICorPM', 'ConvertedCompYearly', 'LogSalary', 'SeniorityYears', 'SeniorityBucket', 'DevTypePrimary', 'DevCategory', 'CountryTier', 'OrgSizeNum']
Dropped 215 rows missing salary or DevType.

Final H1 dataset shape: (156992, 19)

Rows per year:
Year
2020    28816
2021    40145
2022    29737
2023    38862
2024    16979
2025     2453



## 12 . Save cleaned dataset


In [29]:
OUTPUT_PATH = "h1_clean_new.csv"
df_h1.to_csv(OUTPUT_PATH, index=False)
print("Saved data to {}".format(OUTPUT_PATH))

Saved data to h1_clean_new.csv
